# 🤖 Simple RAG Demo — HR Policy Assistant

**RAG** stands for **Retrieval-Augmented Generation**. In plain English:

1. We take a document (here, an HR policy handbook).
2. We chop it into small pieces and turn each piece into a list of numbers (an "embedding") that captures its meaning.
3. We store those pieces in a searchable database (a "vector store").
4. When a user asks a question, we **search** the database for the most relevant pieces, and hand them to an AI model to **generate** a final answer.

That's it. No magic — just search + a language model.

**What we use in this notebook:**
- 📄 Data: `data/hr_policy.txt` (a sample HR policy document)
- 🔢 Embeddings: **Jina AI**
- 🗄️ Vector store: **FAISS**
- 🧠 LLM: **Groq** (fast + free-tier friendly)
- 🕸️ Framework: **LangChain** (`create_agent`)

> Before running: make sure the notebook's kernel is set to the `ragenv` virtual environment (top-right corner of VS Code / Jupyter), and that a `.env` file with `GROQ_API_KEY` and `JINA_API_KEY` exists in this folder.

## Step 1 — Import everything we need

We import all the tools upfront so it's clear what's being used and where it comes from.

In [5]:
import os 
from dotenv import load_dotenv

# langchain
from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter 
from langchain_community.embeddings import JinaEmbeddings



In [6]:
load_dotenv()

True

In [7]:
groq_key = os.getenv("GROQ_API_KEY")
jina_key = os.getenv("JINA_API_KEY")

print("ENV VAR LOADED")

ENV VAR LOADED


LOADING OUR DATA

In [8]:
DATA_FILE_PATH = os.path.join("data" , "hr_policy.txt")

#### DATA INGESTION 

In [9]:
DATA_FILE_PATH = os.path.join("data", "hr_policy.txt")

loader = TextLoader(DATA_FILE_PATH, encoding="utf-8")
documents = loader.load()

print(f"Loaded file: {DATA_FILE_PATH}")
print(f"Number of documents loaded: {len(documents)}")
print(f"Total characters in document: {len(documents[0].page_content)}")
print("\n--- Preview of first 300 characters ---")
print(documents[0].page_content[:300])

Loaded file: data\hr_policy.txt
Number of documents loaded: 1
Total characters in document: 2598

--- Preview of first 300 characters ---
COMPANY HR POLICY HANDBOOK
Acme Corp - Employee Handbook (Demo Document)

1. LEAVE POLICY
All full-time employees are entitled to 20 days of paid annual leave per calendar year.
Leave requests must be submitted through the HR portal at least 5 working days in advance.
Unused annual leave can be carr


#### LANGCHAIN DOCUMENT 

Langchain processes everything in form of documents 


DOCUMENTS : 

PAGE CONTENT -- the actual data 

METADATA  - extra information about the data 

In [10]:
len(documents)

1

In [11]:
print(documents[0].page_content)

COMPANY HR POLICY HANDBOOK
Acme Corp - Employee Handbook (Demo Document)

1. LEAVE POLICY
All full-time employees are entitled to 20 days of paid annual leave per calendar year.
Leave requests must be submitted through the HR portal at least 5 working days in advance.
Unused annual leave can be carried forward to the next year, up to a maximum of 5 days.
Sick leave is separate from annual leave, and employees get 10 paid sick days per year.
A medical certificate is required for sick leave longer than 2 consecutive days.

2. WORK FROM HOME POLICY
Employees may work from home up to 2 days per week, subject to manager approval.
Fully remote work arrangements require written approval from the department head.
Employees working from home must be reachable during core hours: 10 AM to 4 PM.

3. PROBATION PERIOD
All new employees undergo a probation period of 3 months from their date of joining.
During probation, employees are not eligible for paid leave, but may take unpaid leave
in case of e

In [12]:
print(documents[0].metadata)

{'source': 'data\\hr_policy.txt'}


In [13]:
print(f"Total characters in document: {len(documents[0].page_content)}")

Total characters in document: 2598


SPLITTING OUR DATA

In [14]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size = 500,
    chunk_overlap=50
)

chunks = text_splitter.split_documents(documents)

print(chunks)

[Document(metadata={'source': 'data\\hr_policy.txt'}, page_content='COMPANY HR POLICY HANDBOOK\nAcme Corp - Employee Handbook (Demo Document)'), Document(metadata={'source': 'data\\hr_policy.txt'}, page_content='1. LEAVE POLICY\nAll full-time employees are entitled to 20 days of paid annual leave per calendar year.\nLeave requests must be submitted through the HR portal at least 5 working days in advance.\nUnused annual leave can be carried forward to the next year, up to a maximum of 5 days.\nSick leave is separate from annual leave, and employees get 10 paid sick days per year.\nA medical certificate is required for sick leave longer than 2 consecutive days.'), Document(metadata={'source': 'data\\hr_policy.txt'}, page_content='2. WORK FROM HOME POLICY\nEmployees may work from home up to 2 days per week, subject to manager approval.\nFully remote work arrangements require written approval from the department head.\nEmployees working from home must be reachable during core hours: 10 AM

In [15]:
len(chunks)

9

NOW EACH SPILTED CHUNK IS A DOCUMENT - page content and metadata

In [16]:
print(chunks[0])

page_content='COMPANY HR POLICY HANDBOOK
Acme Corp - Employee Handbook (Demo Document)' metadata={'source': 'data\\hr_policy.txt'}


In [17]:
print(chunks[5])

page_content='5. REIMBURSEMENT POLICY
Employees can claim reimbursement for approved business expenses such as travel,
client meals, and internet bills used for official work.
All reimbursement claims must be submitted with valid bills within 30 days of the expense.
Claims are processed within 10 working days after approval from the reporting manager.' metadata={'source': 'data\\hr_policy.txt'}


In [18]:
print(chunks[8].page_content)

8. EXIT POLICY
Upon resignation or termination, employees must complete a clearance process involving
IT, Finance, and HR departments before their last working day.
Full and final settlement, including any pending reimbursements and leave encashment,
is processed within 45 days of the last working day.


In [19]:
print(chunks[7].page_content)

7. HOLIDAYS
The company observes 12 public holidays every year, as per the official holiday calendar
published by HR at the start of each year.
Employees working on a public holiday are eligible for compensatory leave.


In [20]:
print(chunks[6].page_content)

6. CODE OF CONDUCT
Employees are expected to maintain professionalism and respect in the workplace.
Harassment, discrimination, or any form of workplace misconduct will not be tolerated
and may result in disciplinary action, including termination.
All employees must complete an annual Code of Conduct training.


EMBEDD OUR DATA

In [21]:
from langchain_community.embeddings import JinaEmbeddings

embeddings_model = JinaEmbeddings(model_name="jina-embeddings-v2-base-en")

print("EMB MODEL READY THE NAME IS ", embeddings_model.model_name)

EMB MODEL READY THE NAME IS  jina-embeddings-v2-base-en


### STORE DATA IN VECTOR DB

In [22]:
from langchain_community.vectorstores import FAISS 

vector_store = FAISS.from_documents(chunks , embeddings_model)

print("CHUNKS ARE STORED" , vector_store.index.ntotal)

CHUNKS ARE STORED 9


WE NEVER STORED IT 

In [23]:
test_query = "How many sick leaves employees get"

## SIMILARITY SEARCH 

top_matches = vector_store.similarity_search(test_query , k=2)
print(f"Query: {test_query}\n")
for i,match in enumerate(top_matches,start=1):
    print(f"--- Match {i} ---")
    print(match.page_content)
    print()


Query: How many sick leaves employees get

--- Match 1 ---
1. LEAVE POLICY
All full-time employees are entitled to 20 days of paid annual leave per calendar year.
Leave requests must be submitted through the HR portal at least 5 working days in advance.
Unused annual leave can be carried forward to the next year, up to a maximum of 5 days.
Sick leave is separate from annual leave, and employees get 10 paid sick days per year.
A medical certificate is required for sick leave longer than 2 consecutive days.

--- Match 2 ---
7. HOLIDAYS
The company observes 12 public holidays every year, as per the official holiday calendar
published by HR at the start of each year.
Employees working on a public holiday are eligible for compensatory leave.



#### TOOL

In [24]:
retriever = vector_store.as_retriever(search_kwargs={"k": 3})  # returns top 3 relevant chunks

def search_hr_policy(question:str)->str:
    """
    Search the HR policy document for information about leave, work from home,
    probation, notice period, reimbursement, code of conduct, holidays, or exit process.
    
    """
    matching_chunks = retriever.invoke(question)
    return "\n\n".join(chunk.page_content for chunk in matching_chunks)

### DATA RETRIVAL

LLM 

In [25]:
from langchain_groq import ChatGroq

llm = ChatGroq(
    model = "openai/gpt-oss-120b",
    temperature=0.5  # creativity 
)

llm.model_name

'openai/gpt-oss-120b'

In [26]:
tr = llm.invoke("Hey what is the leave policy")

In [27]:
tr.content

'Sure! While the exact details can vary from one organization to another, most companies’ leave policies cover a few common categories. Here’s a typical overview:\n\n| **Leave Type** | **Typical Entitlement** | **Key Points** |\n|----------------|------------------------|----------------|\n| **Annual / Vacation Leave** | 10‑30\u202fdays per year (often prorated for new hires) | Usually accrued monthly; may roll over a limited amount or be “use‑or‑lose.” |\n| **Sick Leave** | 5‑12\u202fdays per year (sometimes separate from vacation) | Often requires a doctor’s note after a certain number of consecutive days. |\n| **Personal / Compassionate Leave** | 1‑5\u202fdays per year | For urgent family matters, bereavement, or other personal emergencies. |\n| **Parental / Maternity/Paternity Leave** | 6‑26\u202fweeks (or more) depending on local law & company policy | May be paid, partially paid, or unpaid; can include adoption leave. |\n| **Public / Statutory Holidays** | As mandated by local la

AI AGENT

3 -- 

LLM - BRAIN 

TOOL - SUPER POWER 

MEMORY - no memory 

In [28]:
from langchain.agents import create_agent      

In [29]:
hr_assistant = create_agent(
    model = llm,
    tools=[search_hr_policy],
    system_prompt= """ 
    
    You are a friendly HR assistant working for Acme Crop. 
    Always use the search_hr_policy tool to look up 
    facts before answering. 
    If the answer isn't in the search results, say you don't know "
    instead of guessing."
    """
)

print("HR assistant agent is ready to answer questions!")

HR assistant agent is ready to answer questions!


In [30]:
def ask_hr_assistant(question: str) -> str:
    """Send a question to the RAG agent and print a nicely formatted answer."""
    print("=" * 60)
    print("QUESTION:", question)
    print("-" * 60)

    response = hr_assistant.invoke({"messages": [{"role": "user", "content": question}]})
    answer = response["messages"][-1].content

    print("ANSWER:", answer)
    print("=" * 60)
    print()
    return answer

In [31]:
response = hr_assistant.invoke(
    {
        "messages":[
            {
                "role":"user",
                "content": "tell me which org you work for"
            }
        ]
    }
)


In [32]:
response 

{'messages': [HumanMessage(content='tell me which org you work for', additional_kwargs={}, response_metadata={}, id='c06ba7b3-4b2c-40d5-86b0-cae022e67527'),
  AIMessage(content='I’m the HR assistant here at **Acme\u202fCrop**.', additional_kwargs={'reasoning_content': 'The user asks: "tell me which org you work for". The assistant is a friendly HR assistant working for Acme Crop. Must answer based on policy? It\'s not about HR policy. The instruction says always use search_hr_policy tool to look up facts before answering. The question is about which organization the assistant works for. That is not in HR policy. So we should say we don\'t know? But we actually know from system: "You are a friendly HR assistant working for Acme Crop." That\'s not HR policy; it\'s given in system. The developer instruction: "You are a friendly HR assistant working for Acme Crop." So we can answer: I work for Acme Crop. No need to search HR policy because not about HR policy. But the instruction says alwa

SYSTEM MESSAGE - HR ASSISNT

HUMAN MESSAGE - TELL ME ABOUT POLICIES 

AI MESSAGE  - HEY THESE ARE THEPLOICES 


In [33]:
response["messages"][-2].content

'tell me which org you work for'

In [34]:
print(response["messages"][-1].content)

I’m the HR assistant here at **Acme Crop**.
